In [1]:
import AILibs
import numpy

from matplotlib import pyplot as plt

dataset_root_path = "/users/michal/datasets/FordA/"

dataset_train   = AILibs.datasets.FordDataset(dataset_root_path, split='TRAIN')
dataset_test    = AILibs.datasets.FordDataset(dataset_root_path, split='TEST')

x_train = dataset_train.features
y_train = dataset_train.labels

x_test = dataset_test.features
y_test = dataset_test.labels

num_classes = dataset_train.num_classes



print("Extracting features")
features_extractor = AILibs.features.Catch22Features(x_train)
#features_extractor = AILibs.features.RocketFeatures(x_train, 1024)

# obtain features for train and test sets
z_train = features_extractor.forward(x_train)   
z_test  = features_extractor.forward(x_test)

#z_train = numpy.reshape(x_train, (x_train.shape[0], -1))
#z_test  = numpy.reshape(x_test, (x_test.shape[0], -1))

print("Feature shape: ", z_train.shape)
print("Test feature shape: ", z_test.shape)

print("Training forest")

forest = AILibs.forest.RandomForest()   


# one hot encoding
y_one_hot = numpy.eye(num_classes)[y_train.astype(int)]

forest.fit(z_train, y_one_hot, max_depth=8, num_trees=256, num_subsamples=512, num_random_candidates=16)
#forest.fit(z_train, y_one_hot, max_depth=10, num_trees=256, num_subsamples=-1, num_random_candidates=16)



Loading FordA TRAIN split...
Loaded 3601 samples.
Feature shape per sample: (500, 1) (seq_length, num_features)
Number of unique classes found: 2
Loading FordA TEST split...
Loaded 1320 samples.
Feature shape per sample: (500, 1) (seq_length, num_features)
Number of unique classes found: 2
Extracting features
Feature shape:  (3601, 22)
Test feature shape:  (1320, 22)
Training forest


In [2]:


print("Predicting with Random Forest...")
y_pred = forest.predict_batch(z_test)


metrics = AILibs.metrics.classification_evaluation(y_test, y_pred,num_classes)


for key, value in metrics.items():
    print(f"{key}: {value}")

    

Predicting with Random Forest...
n_samples: 1320
num_classes: 2
accuracy: 0.90833
macro_precision: 0.90834
macro_recall: 0.90812
macro_f1_score: 0.90822
macro_mcc: 0.81646
macro_specificity: 0.90812
macro_balanced_accuracy: 0.90812
macro_iou: 0.83188
macro_dice: 0.90822
tp_per_class: [623, 576]
tn_per_class: [576, 623]
fp_per_class: [63, 58]
fn_per_class: [58, 63]
precision_per_class: [0.90816, 0.90852]
recall_per_class: [0.91483, 0.90141]
f1_score_per_class: [0.91149, 0.90495]
